In [4]:
!wget "https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/21154/1243559/compressed/tfrecords-jpeg-331x331.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1784514229&Signature=nRRXT6%2FWqkqAFMlEOlZIjjg2T6BuOkMAUslPQQe%2B038MAqLZaKTb2hAuTm28OmuCU2NsiWHuywScepG6Wnmo%2FoNponSMODJWMwPRTaHQKGuBUQe%2BsNvhIb4UH9Ku7%2B5XTomGVn74szotSzgpYUl3uWkEOxUojEAbtVonbSCuGfc2RAXH7m8FIHPIlQEhPd%2FZeonlCJW6UdOG7Hgt71jfNCNYN7q30KID1uxhDaFlsAAGSZJQNOCruMf1IfzBWxs%2FcejOQptEsg%2BLI0qdm8p4NPVpbmxPw0Up4qDiK%2FdiMeJyyIBnFWgCGoiXLRqRa7jKh5sJ4GjCjsXEmx8Wfy2tVw%3D%3D&response-content-disposition=attachment%3B+filename%3Dtfrecords-jpeg-331x331.zip"

The destination name is too long (549), reducing to 236
--2026-07-17 03:50:49--  https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/21154/1243559/compressed/tfrecords-jpeg-331x331.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1784514229&Signature=nRRXT6%2FWqkqAFMlEOlZIjjg2T6BuOkMAUslPQQe%2B038MAqLZaKTb2hAuTm28OmuCU2NsiWHuywScepG6Wnmo%2FoNponSMODJWMwPRTaHQKGuBUQe%2BsNvhIb4UH9Ku7%2B5XTomGVn74szotSzgpYUl3uWkEOxUojEAbtVonbSCuGfc2RAXH7m8FIHPIlQEhPd%2FZeonlCJW6UdOG7Hgt71jfNCNYN7q30KID1uxhDaFlsAAGSZJQNOCruMf1IfzBWxs%2FcejOQptEsg%2BLI0qdm8p4NPVpbmxPw0Up4qDiK%2FdiMeJyyIBnFWgCGoiXLRqRa7jKh5sJ4GjCjsXEmx8Wfy2tVw%3D%3D&response-content-disposition=attachment%3B+filename%3Dtfrecords-jpeg-331x331.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.203.207, 142.250.98.207, 74.125.134.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.203.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Leng

In [5]:
import shutil

# Extract the entire ZIP file into a specific directory
shutil.unpack_archive("tfrecords-jpeg-331x331.zip", "tfrecords-jpeg-331x331")

In [6]:
import tensorflow as tf

In [7]:
print("TF version:", tf.__version__)
strategy = tf.distribute.get_strategy()
print("GPU:", tf.config.list_physical_devices('GPU'))
AUTOTUNE = tf.data.AUTOTUNE

TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
import os

In [9]:
IMAGE_SIZE = [331,331]
NUM_CLASSES = 104
BATCH_SIZE = 16

folder_path = '/content/tfrecords-jpeg-331x331'

TRAIN_FILES = [
    os.path.join(folder_path, "train", f)
    for f in os.listdir(os.path.join(folder_path, "train"))
    if f.endswith(".tfrec")
]

VAL_FILES = [
    os.path.join(folder_path, "val", f)
    for f in os.listdir(os.path.join(folder_path, "val"))
    if f.endswith(".tfrec")
]

print("files:", len(TRAIN_FILES), len(VAL_FILES))

files: 16 16


In [10]:
import re
import numpy as np

# Decode a JPEG string into a normalized float image
def decode_image(image_data):
    image = tf.image.decode_jpeg(image_data, channels=3)
    image = tf.cast(image, tf.float32)
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

# Parse a labeled example (image + class)
def read_labeled(example):
    fmt = {"image": tf.io.FixedLenFeature([], tf.string),
           "class": tf.io.FixedLenFeature([], tf.int64)}
    ex = tf.io.parse_single_example(example, fmt)
    return decode_image(ex["image"]), tf.cast(ex["class"], tf.int32)

# Parse an unlabeled example (image + id)
def read_unlabeled(example):
    fmt = {"image": tf.io.FixedLenFeature([], tf.string),
           "id": tf.io.FixedLenFeature([], tf.string)}
    ex = tf.io.parse_single_example(example, fmt)
    return decode_image(ex["image"]), ex["id"]

# Count images from the number encoded in each filename
def count_items(filenames):
    n = [int(re.compile(r"-([0-9]*)\.").search(f).group(1)) for f in filenames]
    return int(np.sum(n))

In [11]:
# Build a TFRecord dataset, optionally deterministic
def load_dataset(filenames, labeled=True, ordered=False):
    ds = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTOTUNE)
    if ordered:
        opt = tf.data.Options(); opt.deterministic = True
        ds = ds.with_options(opt)
    ds = ds.map(read_labeled if labeled else read_unlabeled, num_parallel_calls=AUTOTUNE)
    return ds

# Training set: shuffled, repeated, batched
def get_training_dataset():
    ds = load_dataset(TRAIN_FILES, labeled=True)
    return ds.repeat().shuffle(2048).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Validation set: ordered, batched
def get_validation_dataset():
    return load_dataset(VAL_FILES, labeled=True, ordered=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)

NUM_TRAIN, NUM_VAL= count_items(TRAIN_FILES), count_items(VAL_FILES)
STEPS_PER_EPOCH = NUM_TRAIN // BATCH_SIZE
print("train/val:", NUM_TRAIN, NUM_VAL, "| steps/epoch:", STEPS_PER_EPOCH)

train/val: 12753 3712 | steps/epoch: 797


In [12]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomContrast(0.20),
    tf.keras.layers.RandomBrightness(0.20),
], name="augmentation")

In [13]:
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(331,331,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(331,331,3))

x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)


x = Dense(512, activation='swish')(x)
x = Dropout(0.4)(x)

x = Dense(256, activation='swish')(x)
x = Dropout(0.4)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [14]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 331, 331, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 331, 331, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 11, 11, 1280)   │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 104)            │        26,728 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,863,499 (18.55 MB)

 Trainable params: 813,928 (3.10 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_acc"
        )
    ]
)

In [16]:
def lrfn(epoch):
  start, mx, mn, warm = 1e-5, 2e-4, 1e-6, 3
  if epoch < warm:
      return start + (mx - start) * epoch / warm
  return mn + (mx - mn) * (0.8 ** (epoch - warm))

In [17]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_efficientnet.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=1)
]

In [18]:
history = model.fit(
    get_training_dataset(),
    validation_data=get_validation_dataset(),
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=NUM_VAL // BATCH_SIZE,
    epochs=15,
    callbacks=callbacks
)


Epoch 1: LearningRateScheduler setting learning rate to 1e-05.
Epoch 1/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 117s 123ms/step - accuracy: 0.0522 - loss: 4.5237 - top5_acc: 0.1530 - val_accuracy: 0.1479 - val_loss: 4.3061 - val_top5_acc: 0.3675 - learning_rate: 1.0000e-05

Epoch 2: LearningRateScheduler setting learning rate to 7.333333333333333e-05.
Epoch 2/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 96s 121ms/step - accuracy: 0.2273 - loss: 3.6188 - top5_acc: 0.4410 - val_accuracy: 0.3753 - val_loss: 2.7107 - val_top5_acc: 0.6794 - learning_rate: 7.3333e-05

Epoch 3: LearningRateScheduler setting learning rate to 0.00013666666666666666.
Epoch 3/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 141s 178ms/step - accuracy: 0.4071 - loss: 2.5287 - top5_acc: 0.6760 - val_accuracy: 0.6175 - val_loss: 1.6405 - val_top5_acc: 0.8467 - learning_rate: 1.3667e-04

Epoch 4: LearningRateScheduler setting learning rate to 0.0002.
Epoch 4/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 97s 122ms/step - accuracy: 0.5178 - loss: 1.9132 - top5_acc: 0.79

In [25]:
base_model.trainable = True

for layer in base_model.layers[:-10]:
  layer.trainable = False

In [26]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_acc"
        )
    ]
)

In [27]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 331, 331, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 331, 331, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 11, 11, 1280)   │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 104)            │        26,728 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,863,499 (18.55 MB)

 Trainable params: 1,707,160 (6.51 MB)

 Non-trainable params: 3,156,339 (12.04 MB)

In [ ]:
history_ft = model.fit(
    get_training_dataset(),
    validation_data=get_validation_dataset(),
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=NUM_VAL // BATCH_SIZE,
    epochs=12,
    callbacks=callbacks
)


Epoch 1: LearningRateScheduler setting learning rate to 1e-05.
Epoch 1/12
797/797 ━━━━━━━━━━━━━━━━━━━━ 121s 130ms/step - accuracy: 0.6788 - loss: 1.3144 - top5_acc: 0.9011 - val_accuracy: 0.8044 - val_loss: 0.7527 - val_top5_acc: 0.9512 - learning_rate: 1.0000e-05

Epoch 2: LearningRateScheduler setting learning rate to 7.333333333333333e-05.
Epoch 2/12
797/797 ━━━━━━━━━━━━━━━━━━━━ 104s 130ms/step - accuracy: 0.7051 - loss: 1.0864 - top5_acc: 0.9188 - val_accuracy: 0.8378 - val_loss: 0.6024 - val_top5_acc: 0.9601 - learning_rate: 7.3333e-05

Epoch 3: LearningRateScheduler setting learning rate to 0.00013666666666666666.
Epoch 3/12
797/797 ━━━━━━━━━━━━━━━━━━━━ 105s 132ms/step - accuracy: 0.7363 - loss: 0.9623 - top5_acc: 0.9341 - val_accuracy: 0.8572 - val_loss: 0.5390 - val_top5_acc: 0.9655 - learning_rate: 1.3667e-04

Epoch 4: LearningRateScheduler setting learning rate to 0.0002.
Epoch 4/12
797/797 ━━━━━━━━━━━━━━━━━━━━ 105s 131ms/step - accuracy: 0.7695 - loss: 0.8290 - top5_acc: 0.